# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iankitthakur/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
!pip -q install duckdb
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")          # the token stays in Secrets, never in the cell
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
print("connected")

connected


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [12]:
cols = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").fetchall()
for c in cols:
    print(c[0], "|", c[1])

report_date | DATE
client_hash_id | VARCHAR
content_hash_id | VARCHAR
client_has_gsc | BOOLEAN
client_has_ga4 | BOOLEAN
gsc_data_available | BOOLEAN
ga4_data_available | BOOLEAN
gsc_impressions | BIGINT
gsc_clicks | BIGINT
gsc_sum_position | BIGINT
gsc_avg_position | DOUBLE
ga4_pageviews | BIGINT
ga4_sessions | BIGINT
ga4_users | BIGINT
ga4_engaged_sessions | BIGINT
ga4_total_engagement_sec | BIGINT
sessions_organic | BIGINT
sessions_direct | BIGINT
sessions_referral | BIGINT
sessions_social | BIGINT
sessions_paid | BIGINT
sessions_ai | BIGINT
ai_chatgpt | BIGINT
ai_perplexity | BIGINT
ai_gemini | BIGINT
ai_copilot | BIGINT
ai_claude | BIGINT
ai_meta | BIGINT
ai_other | BIGINT
scroll_events | BIGINT
month | VARCHAR


In [13]:
con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")
print("march view ready")

march view ready


In [14]:
con.sql("""
SELECT
  COUNT(*) AS n_rows,
  MIN(report_date) AS first_date,
  MAX(report_date) AS last_date,
  COUNT(DISTINCT client_hash_id) AS n_clients,
  COUNT(DISTINCT content_hash_id) AS n_pages
FROM march
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,first_date,last_date,n_clients,n_pages
0,9841378,2026-03-01,2026-03-31,55,331437


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [16]:
con.sql("""
SELECT
  COUNT(*) AS n_rows,
  (SELECT COUNT(*) FROM (SELECT DISTINCT report_date, client_hash_id, content_hash_id FROM march)) AS n_unique_keys
FROM march
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_unique_keys
0,9841378,9841378


In [17]:
con.sql("""
SELECT
  COUNT(*) AS all_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_gsc_available,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_ga4_available
FROM march
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,rows_gsc_available,rows_ga4_available
0,9841378,3611061,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Only about 4% of March rows have GA4 data, so my slice uses Search Console signals only. Anything about on-site behaviour (sessions, engagement) is outside what this slice can tell me

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.